# E02b portable verification (Kaggle)

Проверяет переносимый inference-пакет **Project Morphogenesis, E02b** — уже
закрытую перепись морфопространства (832 развёртки, 12 канонических чекпоинтов
плюс контроль до обучения).

* вход: Kaggle Dataset `mak777777uh/morphogenesis-e02b-portable` (один архив `e02b-portable.zip`) — lite-чекпоинты
  (`model_state_dict` + metadata, ~0.15 МБ вместо 514.5 МБ), `census.json`,
  замороженные центроиды классов, исходники inference и `verify_portable.py`;
* интернет выключен, ничего не устанавливается: `torch`, `numpy`, `scipy` уже
  есть в образе Kaggle; репозиторий не клонируется и не устанавливается из git;
* запускается `verify_portable.py --mode smoke`: 2 обученные модели
  (R-9 seed42, R-9 seed101) и необученная архитектура × 16 сидов (0–15) × 2000 шагов;
* критерий PASS — все классы совпали с `census.json`, а lite-файлы целы
  (ключи, sha256, отсутствие optimizer/pool/state).

`/kaggle/input` и `/kaggle/working` — стандартные пути платформы Kaggle, а не
пути автора. Это не новый эксперимент: гипотезы E02b не пересматриваются, и по
результатам NCA нельзя объявлять «жизнь», «сознание» или «интеллект».

In [ ]:
import glob
import json
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy
import scipy
import torch

print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("cuda runtime:", torch.version.cuda)
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

work = WORK_ROOT / "e02b-portable"
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True, exist_ok=True)

zips = sorted(glob.glob(str(INPUT_ROOT / "**" / "e02b-portable.zip"), recursive=True))
if zips:
    archive = zips[0]
    print("archive:", archive)
    shutil.unpack_archive(archive, str(work))
else:
    manifests = sorted(glob.glob(str(INPUT_ROOT / "**" / "portable_manifest.json"), recursive=True))
    assert manifests, "portable_manifest.json не найден: подключите датасет к kernel"
    dataset_root = Path(manifests[0]).parent
    print("dataset root:", dataset_root)
    shutil.copytree(dataset_root, work, dirs_exist_ok=True)

manifest_path = work / "portable_manifest.json"
assert manifest_path.is_file(), "portable_manifest.json не найден после распаковки"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("package root:", work)
print("experiment:", manifest["experiment"])
print("families:", len(manifest["package"]["families"]))
print("source commit:", manifest["source"]["git_commit"])
print("spec:", manifest["package"]["spec_version"], manifest["package"]["revision"])
print("package bytes:", manifest["package"]["total_bytes"])
print("census sha256:", manifest["datasets"]["census"]["sha256"])


In [ ]:
report_path = work / "verify_report.json"
cmd = [
    sys.executable,
    str(work / "verify_portable.py"),
    "--mode", "smoke",
    "--device", "auto",
    "--out", str(report_path),
]
print(" ".join(cmd))
started = time.time()
proc = subprocess.run(cmd, cwd=str(work), capture_output=True, text=True)
print(proc.stdout[-12000:])
if proc.stderr.strip():
    print("STDERR:", proc.stderr[-4000:])
print("exit code:", proc.returncode, "| wall seconds:", round(time.time() - started, 1))
assert proc.returncode == 0, "verify_portable.py завершился с ненулевым кодом"
assert report_path.is_file(), "verify_report.json не создан"

In [ ]:
report = json.loads(report_path.read_text(encoding="utf-8"))
summary = report["summary"]
versions = report["versions"]

print("mode:", report["mode"], "| device:", report["device"])
print("torch:", versions["torch"], "| cuda:", versions["cuda"], "| gpu:", versions["gpu"])
print("python:", versions["python"], "| spec:", versions["spec_version"])
print("manifest sha256:", report["package"]["manifest_sha256"])
print("census sha256:", report["census"]["sha256"], "| runs:", report["census"]["n_runs"])
print("runs checked:", summary["n_runs"], "| mismatches:", summary["n_mismatches"])
print("max descriptor diff:", summary["max_abs_descriptor_diff"])
print("rollout seconds:", summary["total_rollout_seconds"], "| wall seconds:", report["wall_seconds"])
print("lite checks ok:", report["checks_ok"])
for check in report["checks"]:
    if check["ok"]:
        print("- lite ok:", check["family"], check["size_bytes"], "bytes")
    else:
        print("- lite FAILED:", check["family"], check["problems"])

failures = [row for row in report["runs"] if not row["class_match"]]
print("class mismatches:", len(failures))
for row in failures[:20]:
    print("  ", row["family"], "seed", row["seed"], "expected", row["expected_class"], "got", row["assigned_class"])

print("\n" + "=" * 64)
print("VERDICT:", report["status"])
print("=" * 64)
assert report["status"] == "PASS", "portable package verification FAILED: " + report["status"]